# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:

orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

import pandas as pd


 **One row of my result is one:** _..._ order line item/individual item purchased in an order



### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total


# TODO 1

lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

assert len(lines) == 4
assert lines['qty'].sum() == 7

print(lines)
# 4 rows printed and quantities of 2+2+3=7

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO


vendors = pd.json_normalize(vendors_json)

joined = lines.merge(
    vendors,
    on='vendor_id',
    how='left',
    indicator=True
)

unmatched = joined[joined['_merge'] == 'left_only']

print(unmatched)
print(unmatched['qty'].sum())

#The unmatched vendor is V-07, with 3 units attached to it.

      item  qty order vendor_id name zone     _merge
3  Hot Dog    3     3      V-07  NaN  NaN  left_only
3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO

# Keep the unmatched vendor as its own row or NaN so its order quantity is not lost.
# groupby drops NaN keys by default, which would lead to losing 3 units(the order from V-07) and make the totals disagre with the 7 units from Q1.
# With dropna=False the table still sums to 7, and the NaN row shows that some units couldn't be matched to a vendor.
quantity_by_vendor = (
    joined.groupby('name', dropna=False)['qty']
    .sum()
    .reset_index()
)

print(quantity_by_vendor)



           name  qty
0     Cav Merch    1
1  Hoos Burgers    3
2           NaN    3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
# Zone A sold the most items, with 4 units. The known vendor zone totals add up to 4, not 7,
# because 3 units belong to the unmatched vendor V-07, which has no zone (NaN). But counting that NaN row, the table still sums to 7, matcheing Q1
zone_totals = (
    joined.groupby('zone', dropna=False)['qty']
    .sum()
    .reset_index()
)

print(zone_totals)


print(zone_totals.loc[zone_totals['qty'].idxmax()])


  zone  qty
0    A    4
1  NaN    3
zone    A
qty     4
Name: 0, dtype: object


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO

# Keep only orders that have a lines list
for order in ragged_json:
    if 'lines' not in order:
        order['lines'] = []

ragged_orders = pd.json_normalize(
    ragged_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

# A missing qty stays NaN. This column marks whether qty was given.
ragged_orders['qty_given'] = ragged_orders['qty'].notna()

print("Order 5 has no line items so it does not appear below.")

print(ragged_orders)



Order 5 has no line items so it does not appear below.
    item  qty order vendor_id  qty_given
0   Soda  2.0     4      V-01       True
1  Fries  NaN     6      V-01      False


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a)
One row per line item is useful because it lets us easily analyze individual items and their quantities across orders. For example, we can calculate the total number of units sold for each vendor or zone. With one row per order, multiple items would be stored together, making it harder to group and sum quantities for individual items. From a coding aspect as well it is easier to implement functions such as groupby for vendors or zones.

b) A missing quantity means we do not know how many units were ordered, while a quantity of zero means we know zero units were ordered. If we treat missing values as zero, we could inaccruately estimate sales and incorrectly conclude that no items were sold when the quantity was actually unknown. This could affect totals, vendor comparisons, and other analysis.